# Experiment 0.2 — Endpoint Tail Regularization

Analysis-only notebook. It reads finalized Exp0.2 artifacts; training and Slurm submission live in the experiment scripts.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO_ROOT = Path('..').resolve()
ARTIFACT_ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_0_2_endpoint_tail_regularization' / 'endpoint_tail_regularization_v1'
DISPLAY_SEED = 11
SAMPLE_INDEX = 4


In [ ]:
manifest = json.loads((ARTIFACT_ROOT / 'manifest.json').read_text())
summary = pd.read_csv(ARTIFACT_ROOT / 'summary.csv')
comparison = pd.read_csv(ARTIFACT_ROOT / 'comparison_summary.csv')
paired = pd.read_csv(ARTIFACT_ROOT / 'paired_vs_frozen_exp01.csv')
calibration = pd.read_csv(ARTIFACT_ROOT / 'calibration_summary.csv')
history = pd.read_csv(ARTIFACT_ROOT / 'history_long.csv')
raster_index = pd.read_csv(ARTIFACT_ROOT / 'raster_index.csv')
manifest


## Classification / settling tradeoff

The primary view keeps test balanced accuracy and final-hidden settling behavior visible together.

In [ ]:
display_cols = [
    'architecture', 'objective', 'profile', 'seed',
    'test_balanced_accuracy', 'test_valid_hidden_activity',
    'test_tail_area_final_hidden', 'test_settling_ms_final_hidden_mean',
]
summary[display_cols].sort_values(['architecture', 'objective', 'profile', 'seed'])


In [ ]:
for (architecture, objective), frame in summary.groupby(['architecture', 'objective']):
    means = frame.groupby('profile', as_index=False)[['test_tail_area_final_hidden', 'test_balanced_accuracy']].mean()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(means['test_tail_area_final_hidden'], means['test_balanced_accuracy'])
    for row in means.itertuples(index=False):
        ax.annotate(row.profile, (row.test_tail_area_final_hidden, row.test_balanced_accuracy))
    ax.set_xlabel('Final-hidden tail area (spikes / neuron)')
    ax.set_ylabel('Test balanced accuracy')
    ax.set_title(f'{architecture} | {objective}')
    plt.show()


## Paired deltas versus frozen Exp0.1


In [ ]:
paired.groupby(['architecture', 'objective', 'profile'])[[
    'delta_test_ba_vs_frozen_exp01',
    'delta_tail_area_vs_frozen_exp01',
    'delta_settling_ms_vs_frozen_exp01',
]].agg(['mean', 'std'])


## Gradient calibration


In [ ]:
calibration.groupby(['architecture', 'objective', 'profile'])[[
    'calibration_ratio_median', 'kappa'
]].agg(['mean', 'std'])


## Fixed-sample firing rasters

All panels use test sample index 4 and include the valid window plus the fixed endpoint-relative zero-input rollout.

In [ ]:
view = raster_index[raster_index['seed'] == DISPLAY_SEED].sort_values(['architecture', 'objective', 'profile'])
for row in view.itertuples(index=False):
    print(row.architecture, row.objective, row.profile, 'seed', row.seed)
    display(Image(filename=str(ARTIFACT_ROOT / row.raster_png)))
